# 🏦 Data Designer Tutorial: AML Case Generation

#### 📚 What you'll learn

This notebook demonstrates the basics of Data Designer by generating an "Anti-Money Laundering (AML)" case dataset.

Example scenario flow:
- Customer opens account
  ↓
- Large incoming transfers
  ↓
- Multiple cash withdrawals
  ↓
- Foreign transfer
  ↓
- PEP relationship
  ↓
- SAR Report

Data Designer can generate:
- Different countries
- Different amounts
- Different occupations
- Different risk levels (Low, Medium, High)
- Different sources of fund (Salary, Crypto, Casino, Inheritance, Business Income, Gift, Unknown)

This is a highly demanded dataset for banking compliance and fraud detection teams.

#### 📚 学習内容

このノートブックでは、「マネーロンダリング対策（AML）」ケースデータセットを生成することで、Data Designerの基本操作を解説します。

シナリオ例：
- 顧客が口座を開設

↓
- 高額の入金

↓
- 複数回の現金引き出し

↓
- 海外送金

↓
- PEP（政治的に重要な人物）との関係

↓
- SAR（疑わしい取引報告書）

Data Designerで生成できるデータ：
- 異なる国
- 異なる金額
- 異なる職業
- 異なるリスクレベル（低、中、高）
- 異なる資金源（給与、仮想通貨、カジノ、相続、事業収入、贈与、不明）

これは、銀行のコンプライアンスおよび不正検出チームにとって非常に需要の高いデータセットです。

### 📦 Import Data Designer

- `data_designer.config` provides access to the configuration API.
- `DataDesigner` is the main interface for data generation.

日本語: `data_designer.config` は設定APIへのアクセスを提供し、`DataDesigner` はデータ生成の主要インターフェースです。

In [1]:
# ! export NVIDIA_API_KEY="" # TODO copy your nvidia-api-key here:
# https://build.nvidia.com/settings/api-keys
# login with your email box (or register)
# click "Generate API key"
import os

os.environ["NVIDIA_API_KEY"] = "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" # TODO replace with your true api key!!!
print(os.getenv("NVIDIA_API_KEY"))

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


In [2]:
# ! pip install data-designer

In [3]:
import data_designer.config as dd
from data_designer.interface import DataDesigner

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### ⚙️ Initialize the Data Designer interface

- `DataDesigner` is the main object responsible for managing the data generation process.
- When initialized without arguments, the default model providers are used.

日本語: `DataDesigner` はデータ生成プロセスを管理する主要オブジェクトです。引数なしで初期化するとデフォルトのモデルプロバイダが使用されます。

In [4]:
data_designer = DataDesigner()

### 🎛️ Define model configurations

- Each `ModelConfig` defines a model that can be used during generation.
- The "model alias" is used to reference the model in the Data Designer config.
- The "model provider" is the external service that hosts the model (default: build.nvidia.com).

日本語: 各 `ModelConfig` は生成時に使用できるモデルを定義します。「モデルエイリアス」は設定内でモデルを参照するために使用され、「モデルプロバイダ」はモデルをホストする外部サービスです（デフォルトは build.nvidia.com）。

In [5]:
MODEL_PROVIDER = "nvidia"
MODEL_ID = "nvidia/nemotron-3-nano-30b-a3b"
MODEL_ALIAS = "nemotron-nano-v3"

model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_ID,
        provider=MODEL_PROVIDER,
        inference_parameters=dd.ChatCompletionInferenceParams(
            temperature=1.0,
            top_p=1.0,
            max_tokens=2048,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        ),
    )
]

### 🏗️ Initialize the Data Designer Config Builder

- The config builder provides an intuitive interface for building the dataset schema and generation process.
- The list of model configs is provided at initialization.

日本語: 設定ビルダーはデータセットスキーマと生成プロセスを構築するための直感的なインターフェースを提供します。モデル設定のリストは初期化時に渡します。

In [6]:
config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)

## 🎲 Sampler columns: AML case attributes

We'll define sampler columns for:
- Customer age
- Country (where the account is held / transaction originates)
- Occupation
- Source of fund (Salary, Crypto, Casino, Inheritance, Business Income, Gift, Unknown)
- Transaction amount (USD)
- PEP (Politically Exposed Person) relationship flag
- Number of cash withdrawals
- Foreign transfer flag
- AML Risk Level (Low, Medium, High) – weighted to reflect real-world distribution

These samplers will drive diversity in the generated AML narratives and SAR reports.

日本語: サンプラー列を定義して、年齢、国、職業、資金源、取引金額、PEPフラグ、現金引出回数、海外送金フラグ、AMLリスクレベルなどの多様性を確保します。

In [7]:
# Customer age (18-80)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="age",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=18, high=80),
        convert_to="int",
    )
)

# Country
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="country",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "United States", "United Kingdom", "Canada", "Australia", "Germany",
                "France", "Spain", "Italy", "Japan", "Brazil", "Mexico", "India",
                "South Africa", "Nigeria", "Singapore", "UAE", "Russia", "China",
                "Switzerland", "Cayman Islands", "Panama"
            ]
        ),
    )
)

# Occupation
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="occupation",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Business Owner", "Corporate Executive", "Software Engineer",
                "Doctor", "Lawyer", "Accountant", "Teacher", "Student",
                "Retired", "Government Official", "Military Personnel",
                "Freelancer", "Trader", "Real Estate Agent", "Politician"
            ]
        ),
    )
)

# Source of Fund
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="source_of_fund",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=[
                "Salary", "Crypto", "Casino", "Inheritance",
                "Business Income", "Gift", "Unknown"
            ]
        ),
    )
)

# Transaction amount (USD) - large range to simulate AML thresholds
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="transaction_amount",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=1000, high=1000000),
        convert_to="int",
    )
)

# PEP (Politically Exposed Person) relationship flag (10% chance to simulate rare cases)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="is_pep",
        sampler_type=dd.SamplerType.BERNOULLI,
        params=dd.BernoulliSamplerParams(p=0.1),
        convert_to="int",
    )
)

# Number of cash withdrawals (0 to 20)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="num_cash_withdrawals",
        sampler_type=dd.SamplerType.UNIFORM,
        params=dd.UniformSamplerParams(low=0, high=20),
        convert_to="int",
    )
)

# Foreign transfer flag (30% chance)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="has_foreign_transfer",
        sampler_type=dd.SamplerType.BERNOULLI,
        params=dd.BernoulliSamplerParams(p=0.3),
        convert_to="int",
    )
)

# AML Risk Level (weighted: Low 40%, Medium 35%, High 25%)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="aml_risk_level",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Low", "Medium", "High"],
            weights=[0.4, 0.35, 0.25]
        ),
    )
)

# Language for the conversation (we'll generate in multiple languages)
config_builder.add_column(
    dd.SamplerColumnConfig(
        name="language",
        sampler_type=dd.SamplerType.CATEGORY,
        params=dd.CategorySamplerParams(
            values=["Japanese"] #["English", "Spanish", "French", "German", "Japanese", "Portuguese", "Hindi"]
        ),
    )
)

# Optionally validate
data_designer.validate(config_builder)

[22:42:08] [INFO] ✅ Validation passed


## 🦜 LLM-generated AML narrative columns

We'll generate two columns:
- `case_timeline`: a narrative describing the account opening, incoming transfers, cash withdrawals, foreign transfers, PEP relationship, and the resulting SAR filing.
- `sar_report`: a formal Suspicious Activity Report (SAR) summary drafted by a financial intelligence officer.

We use Jinja templating to reference all sampler columns (age, country, occupation, source_of_fund, amount, is_pep, withdrawals, foreign transfer, risk level).

日本語: LLMを用いてAML調査のケースタイムラインと正式なSARレポートを生成します。Jinjaテンプレートでサンプラー列を参照し、リスクレベルや各種フラグに応じて内容を変化させます。

In [8]:
# Case timeline narrative
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="case_timeline",
        prompt=(
            "You are a compliance officer writing a case summary for an AML investigation. "
            "The customer is aged {{ age }}, from {{ country }}, occupation {{ occupation }}. "
            "Source of funds is {{ source_of_fund }}. "
            "Total transaction amount is ${{ transaction_amount }}. "
            "PEP relationship: {{ is_pep }} (1 = yes, 0 = no). "
            "Number of cash withdrawals: {{ num_cash_withdrawals }}. "
            "Foreign transfer occurred: {{ has_foreign_transfer }} (1 = yes, 0 = no). "
            "AML Risk Level: {{ aml_risk_level }}. "
            "Write a concise professional narrative (4-6 sentences) describing the entire case flow: "
            "account opening, large incoming transfers, cash withdrawal patterns, foreign transfer details, "
            "PEP relationship (if any), the red flags identified, and the final decision to file a SAR. "
            "Do not add meta-commentary; only output the case timeline."
            "The message should be in {{ language }}. "
        ),
        model_alias=MODEL_ALIAS,
    )
)

# SAR Report summary
config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="sar_report",
        prompt=(
            "You are a financial intelligence officer drafting a Suspicious Activity Report (SAR). "
            "Customer profile: age {{ age }}, country {{ country }}, occupation {{ occupation }}, "
            "source of funds {{ source_of_fund }}. "
            "Transaction amount: ${{ transaction_amount }}. "
            "PEP status: {{ is_pep }}. Cash withdrawals: {{ num_cash_withdrawals }}. "
            "Foreign transfer flag: {{ has_foreign_transfer }}. "
            "Overall AML risk level: {{ aml_risk_level }}. "
            "Write a formal SAR summary (3-5 sentences) that clearly lists the red flags observed, "
            "the suspicious activity pattern, and the recommended action (e.g., file SAR, monitor, close account). "
            "Do not add meta-commentary; only output the SAR report."
            "The message should be in {{ language }}. "
        ),
        model_alias=MODEL_ALIAS,
    )
)

data_designer.validate(config_builder)

[22:42:08] [INFO] ✅ Validation passed


### 🔁 Preview the dataset

Generate a small sample to verify quality and format.

日本語: 少数のサンプルを生成して品質を確認します。

In [9]:
preview = data_designer.preview(config_builder, num_records=3)

[22:42:08] [INFO] 🔁 Preview generation in progress
[22:42:08] [INFO]   |-- 🔒 Jinja rendering engine: secure
[22:42:08] [INFO] ✅ Validation passed
[22:42:08] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[22:42:08] [INFO] 🩺 Running health checks for models...
[22:42:08] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[22:42:08] [INFO]   |-- ✅ Passed!
[22:42:08] [INFO] ⚡ Using async task-queue preview
[22:42:08] [INFO] 📝 llm-text model config for column 'case_timeline'
[22:42:08] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[22:42:08] [INFO]   |-- model alias: 'nemotron-nano-v3'
[22:42:08] [INFO]   |-- model provider: 'nvidia'
[22:42:08] [INFO]   |-- inference parameters:
[22:42:08] [INFO]   |  |-- generation_type=chat-completion
[22:42:08] [INFO]   |  |-- max_parallel_requests=4
[22:42:08] [INFO]   |  |-- extra_body={'chat_template_kwargs': {'enable_thinking': False}}
[22:42:08] [INFO]   |

In [10]:
# Display one record at a time
preview.display_sample_record()

                                              Generated Columns                                               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Name                 ┃ Value                                                                               ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ age                  │ 30                                                                                  │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ country              │ Mexico                                                                              │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ occupation           │ Corporate Executive                                                                 │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ source_of_fund       │ Casino                                                                              │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ transaction_amount   │ 620097                                                                              │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ is_pep               │ 0                                                                                   │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ num_cash_withdrawals │ 13                                                                                  │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ has_foreign_transfer │ 0                                                                                   │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ aml_risk_level       │ Low                                                                                 │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ language             │ Japanese                                                                            │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ case_timeline        │ 口座開設後、顧客（30歳・メキシコ国籍・企業経営者）がカジノ経営からの資金に関連する… │
│                      │ 현금                                                                                │
│                      │ 인출を行い、資金を分割受け取りました。外為送金は一切行われず、PEP関係はなし（リス … │
├──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────┤
│ sar_report           │ 顧客（30歳・メキシコ在住・法人エグゼクティブ・資金來源はカジノ）は、$620,097        │
│                      │ の取引額に対して複数回の現金引き出し（13回）を行い、外部送金は検出されておらず、AM… │
│                      │ 本件については、即時 SAR                                                            │
│                      │ の提出並びに後続取引の厳密なモニタリングを実施し、異常が継続する場合はアカウントの… │
└──────────────────────┴─────────────────────────────────────────────────────────────────────────────────────┘

In [11]:
# Show as DataFrame
preview.dataset

,age,country,occupation,source_of_fund,transaction_amount,is_pep,num_cash_withdrawals,has_foreign_transfer,aml_risk_level,language,sar_report,case_timeline
0,30,Mexico,Corporate Executive,Casino,620097,0,13,0,Low,Japanese,"顧客（30歳・メキシコ在住・法人エグゼクティブ・資金來源はカジノ）は、$620,097 の取...",口座開設後、顧客（30歳・メキシコ国籍・企業経営者）がカジノ経営からの資金に関連する複数の大...
1,51,Russia,Doctor,Business Income,812856,1,17,1,Low,Japanese,本件は、51歳のロシア人医師（ビジネス収入あり、PEPあり）がCash withdrawal...,2023年8月15日、顧客（ロシア人、医師、証券口座開設）が口座開設申請。同年9月5日から1...
2,71,Brazil,Politician,Inheritance,310124,0,8,0,High,Japanese,本件では、71歳（ブラジル出身）政治家である顧客が、遺産の源泉からの大規模現金引き出し（8件...,アカウントは71歳のブラジル在住の政治家（PEP関係なし）が開設した。入金額が1件で$310...


### 📊 Analyze the generated data

Data Designer automatically generates basic statistics.

日本語: Data Designerは自動的に基本統計を生成します。

In [12]:
preview.analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 3                               │ 12                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                        ┃        data type ┃              number unique values ┃        sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ age                                │              int │                        3 (100.0%) │             uniform │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ country                            │           string │                        3 (100.0%) │            category │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ occupation                         │           string │                        3 (100.0%) │            category │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ source_of_fund                     │           string │                        3 (100.0%) │            category │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ transaction_amount                 │              int │                        3 (100.0%) │             uniform │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ is_pep                             │              int │                         2 (66.7%) │           bernoulli │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ num_cash_withdrawals               │              int │                        3 (100.0%) │             uniform │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ has_foreign_transfer               │              int │                         2 (66.7%) │           bernoulli │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ aml_risk_level                     │           string │                         2 (66.7%) │            category │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ language                           │           string │                         1 (33.3%) │            category │
└────────────────────────────────────┴──────────────────┴───────────────────────────────────┴─────────────────────┘
                                                         

### 🆙 Scale up!

Once satisfied, generate a larger dataset.

日本語: 満足したら、より大規模なデータセットを生成します。

In [13]:
results = data_designer.create(config_builder, num_records=10, dataset_name="aml_cases")

[22:42:12] [INFO] 🎨 Creating Data Designer dataset
[22:42:12] [INFO]   |-- 🔒 Jinja rendering engine: secure
[22:42:12] [INFO] 📂 Dataset path '/workspace/asr/brev.nemo.curator.20260324/data_designer/artifacts/aml_cases' already exists. Dataset from this session
		     will be saved to '/workspace/asr/brev.nemo.curator.20260324/data_designer/artifacts/aml_cases_07-02-2026_224212' instead.
[22:42:12] [INFO] ✅ Validation passed
[22:42:12] [INFO] ⛓️ Sorting column configs into a Directed Acyclic Graph
[22:42:12] [INFO] 🩺 Running health checks for models...
[22:42:12] [INFO]   |-- 👀 Checking 'nvidia/nemotron-3-nano-30b-a3b' in provider named 'nvidia' for model alias 'nemotron-nano-v3'...
[22:42:12] [INFO]   |-- ✅ Passed!
[22:42:12] [INFO] ⚡ Using async task-queue builder
[22:42:12] [INFO] 📝 llm-text model config for column 'case_timeline'
[22:42:12] [INFO]   |-- model: 'nvidia/nemotron-3-nano-30b-a3b'
[22:42:12] [INFO]   |-- model alias: 'nemotron-nano-v3'
[22:42:12] [INFO]   |-- model provi

In [14]:
dataset = results.load_dataset()
dataset.head()

,age,country,occupation,source_of_fund,transaction_amount,is_pep,num_cash_withdrawals,has_foreign_transfer,aml_risk_level,language,sar_report,case_timeline
0,41,United Kingdom,Politician,Business Income,109119,0,14,0,Medium,Japanese,"偏りのある高額の取引金額（109,119㌦）と14件の大量現金引き出しが観察され、外部への資...",アカウント開設後、41歳の英国男性政財務担当者（非PEP）に対し大額の事業収入入金が14件（...
1,59,Mexico,Government Official,Salary,788206,0,9,0,Medium,Japanese,疑わしい取引の概要：59歳の政府関係者（メキシコ在住）が、給与受取口座から2件の大額現金引き...,"政府関係者が登録した口座に、月額平均$55,500,000千の入金が月数回継続して行われた。..."
2,71,Panama,Corporate Executive,Crypto,79355,0,3,0,Low,Japanese,年齢71歳、パナマ出身の企業エグゼクティブが、暗号資産由来の資金から3件の大額現金引き出しを...,1. アカウントは、71歳のパナマ在住の企業経営者として正規に登録され、投資家向けサービスが...
3,19,Brazil,Teacher,Casino,422668,0,16,0,Medium,Japanese,"年齢19歳のブラジル人教師（PEP:0）で、資金の源泉がカジノである顧客が$422,668を...",口座開設時に、未成熟な学生としての職業（教師）とブラジル出身でありながら、入金合計金額が22...
4,67,Brazil,Doctor,Crypto,710764,0,6,1,High,Japanese,本件は、67歳のブラジル在住医師（PEPではない）が、暗号資産由来の資金から1件の大額取引（...,本件は、ブラジル人医師（年齢67歳）がアカウントを開設し、暗号資産取引による入金（合計710...


In [15]:
analysis = results.load_analysis()
analysis.to_report()

──────────────────────────────────────── 🎨 Data Designer Dataset Profile ─────────────────────────────────────────

                                                                                                                   
                                                 Dataset Overview                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ number of records               ┃ number of columns               ┃ percent complete records                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 10                              │ 12                              │ 100.0%                                      │
└─────────────────────────────────┴─────────────────────────────────┴─────────────────────────────────────────────┘
                                                                                                                   
                                                                                                                   
                                                🎲 Sampler Columns                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ column name                        ┃        data type ┃              number unique values ┃        sampler type ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ age                                │              int │                         9 (90.0%) │             uniform │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ country                            │           string │                         8 (80.0%) │            category │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ occupation                         │           string │                         6 (60.0%) │            category │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ source_of_fund                     │           string │                         6 (60.0%) │            category │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ transaction_amount                 │              int │                       10 (100.0%) │             uniform │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ is_pep                             │              int │                         2 (20.0%) │           bernoulli │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ num_cash_withdrawals               │              int │                         7 (70.0%) │             uniform │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ has_foreign_transfer               │              int │                         2 (20.0%) │           bernoulli │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ aml_risk_level                     │           string │                         3 (30.0%) │            category │
├────────────────────────────────────┼──────────────────┼───────────────────────────────────┼─────────────────────┤
│ language                           │           string │                         1 (10.0%) │            category │
└────────────────────────────────────┴──────────────────┴───────────────────────────────────┴─────────────────────┘
                                                         

In [16]:
dataset.to_csv('aml-01.csv', index=False)

## ⏭️ Next Steps

Now that you've generated an AML case dataset, explore more advanced features:

- [Structured outputs and jinja expressions](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/2-structured-outputs-and-jinja-expressions/)
- [Seeding with an external dataset](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/3-seeding-with-a-dataset/)
- [Providing images as context](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/4-providing-images-as-context/)
- [Generating images](https://nvidia-nemo.github.io/DataDesigner/latest/notebooks/5-generating-images/)

日本語: AMLケースデータセットを生成したので、さらに高度な機能を試してみましょう。